# Demo: Zero-Shot Forecasting with a Foundation Model

Chronos-2 was pretrained on millions of time series across every domain imaginable. It has never seen our subscriber data, but it recognizes the S-curve growth pattern from its training. No `.fit()` call needed -- we just hand it the data and ask for a forecast.

In [ ]:
import os
import warnings
os.environ["HF_HOME"] = os.path.expanduser("~/.cache/huggingface")
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from darts import TimeSeries
from darts.models import ARIMA as DartsARIMA
from darts.metrics import mae, rmse, mape

HORIZON = 12

In [ ]:
df = pd.read_csv("../data/subscribers.csv", parse_dates=["date"], index_col="date").asfreq("MS")
subscribers = df["subscribers"]
ts = TimeSeries.from_series(subscribers)
train, test = ts.split_after(pd.Timestamp("2023-12-01"))
print(f"Train: {len(train)} months, Test: {len(test)} months")

## Running Chronos-2

No training. No hyperparameter tuning. We load the model, pass our series, and get a probabilistic forecast. The `num_samples=100` gives us 100 possible futures so we can estimate uncertainty.

In [ ]:
try:
    from darts.models import Chronos2Model
    chronos = Chronos2Model(model_name="amazon/chronos-bolt-small")
    chronos_fc = chronos.predict(HORIZON, series=train, num_samples=100)
    print("Chronos-2 forecast generated.")
except ImportError:
    print("Chronos-2 not available. Using ARIMA fallback.")
    fallback = DartsARIMA(p=2, d=1, q=1)
    fallback.fit(train)
    chronos_fc = fallback.predict(HORIZON)

## ARIMA Baseline

In [ ]:
arima = DartsARIMA(p=2, d=1, q=1)
arima.fit(train)
arima_fc = arima.predict(HORIZON)

## Comparing

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
ts.plot(ax=ax, label="Actual", color="black", linewidth=1.5)
arima_fc.plot(ax=ax, label="ARIMA(2,1,1)", color="tab:blue")
if chronos_fc.n_samples > 1:
    chronos_fc.plot(ax=ax, label="Chronos-2", color="tab:green", low_quantile=0.05, high_quantile=0.95)
else:
    chronos_fc.plot(ax=ax, label="Chronos-2 (fallback)", color="tab:green")
ax.set_ylabel("Subscribers")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

## Backtesting

A single forecast isn't enough to evaluate a model. Backtesting generates historical forecasts -- slide the training window forward, forecast, compare to actual, repeat. This gives you error metrics across multiple time points.

In [ ]:
full = TimeSeries.from_series(subscribers)
bt = arima.historical_forecasts(full, start=0.75, forecast_horizon=12, stride=1, retrain=False)

print(f"MAE:  {mae(full, bt):,.0f}")
print(f"RMSE: {rmse(full, bt):,.0f}")
print(f"MAPE: {mape(full, bt):.1f}%")

The foundation model approach changes the workflow. With ARIMA or N-BEATS, you spend time selecting model parameters and training. With Chronos-2, you spend zero time on that -- the model already knows general time-series patterns. The question is whether that general knowledge is good enough for your specific data, or whether a trained model would do better. For short series where you don't have enough data to train well, the foundation model often wins.